# 02 - Hipotesis

Fuente: datos.gov.co, Resultados unicos Saber 11, periodo 20224.

Ademas del valor p, en cada prueba reportamos el tamanio del efecto. Con
muestras grandes casi cualquier diferencia sale significativa, entonces el
valor p solo dice si la diferencia existe; el tamanio del efecto dice si
importa.

## Hipotesis 1: puntaje global por zona
- H0: el puntaje global promedio es igual en zona rural y urbana
- H1: el puntaje global promedio es distinto segun la zona

Usamos la prueba t de Welch (no asume varianzas iguales) porque los dos
grupos tienen tamanios muy diferentes. Tamanio del efecto: d de Cohen.

## Hipotesis 2: zona e internet
- H0: la zona no tiene relacion con tener internet
- H1: la zona si tiene relacion con tener internet

Solo entran los estudiantes que respondieron si o no. Los "No informado"
quedan por fuera de esta prueba. Tamanio del efecto: V de Cramer.

## Hipotesis 3: internet y puntaje global
- H0: el puntaje global promedio es igual con y sin internet en el hogar
- H1: el puntaje global promedio es distinto con y sin internet

Es el eslabon que une las dos primeras: si la zona se relaciona con el
internet, falta ver si el internet se relaciona con el puntaje. Prueba t de
Welch y d de Cohen, solo con quienes respondieron si o no.

## Hipotesis 4: estrato y puntaje global
- H0: el puntaje global tiene la misma distribucion en todos los estratos
- H1: al menos un estrato tiene un puntaje distinto

Sirve para revisar si la brecha rural-urbana puede ser en parte un efecto
socioeconomico. Son seis grupos de tamanios muy distintos (estrato 6 tiene
34 estudiantes), por eso usamos Kruskal-Wallis, que no asume normalidad.
Entran solo los estratos 1 a 6; "No informado" y "Sin Estrato" quedan fuera.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from estadistica import (prueba_t, prueba_chi2, d_cohen, v_cramer,
                         prueba_kruskal)

tabla = pd.read_csv('../data/datos_limpios.csv')

In [2]:
tabla.head()

,genero,zona,depto,municipio,estrato,educacion_madre,educacion_padre,internet,matematicas,lectura,global,periodo
0,F,URBANO,CAQUETA,EL DONCELLO,Estrato 2,Primaria incompleta,Primaria incompleta,No,44,50,237,20224
1,F,URBANO,QUINDIO,ARMENIA,Estrato 2,Educación profesional completa,No Aplica,Si,42,40,214,20224
2,M,RURAL,ANTIOQUIA,ANDES,Estrato 2,Primaria incompleta,Primaria incompleta,No,58,59,278,20224
3,F,URBANO,ANTIOQUIA,BELLO,Estrato 3,Educación profesional completa,No sabe,Si,53,47,254,20224
4,M,URBANO,VALLE,PALMIRA,Estrato 2,Técnica o tecnológica completa,No sabe,Si,71,69,341,20224


In [3]:
rural = tabla[tabla['zona'] == 'RURAL']["global"]
urbana = tabla[tabla['zona'] == 'URBANO']["global"]
print('Estudiantes rurales:', len(rural), '| urbanos:', len(urbana))
print('Promedio rural:', round(rural.mean(), 1))
print('Promedio urbana:', round(urbana.mean(), 1))
p1 = prueba_t(rural, urbana)
d1 = d_cohen(urbana, rural)

Estudiantes rurales: 691 | urbanos: 2309
Promedio rural: 224.8
Promedio urbana: 253.5
Estadistico t: -13.678
Valor p < 0.001
Se rechaza H0: si hay diferencia entre los grupos
d de Cohen: 0.548 (efecto mediano)


In [4]:
con_internet = tabla[tabla['internet'].isin(['Si', 'No'])]
cruzada = pd.crosstab(con_internet['zona'], con_internet['internet'])
display(cruzada)
print('Estudiantes que respondieron:', len(con_internet))
p2 = prueba_chi2(cruzada)
v2 = v_cramer(cruzada)

internet,No,Si
zona,,
RURAL,351,261
URBANO,486,1460


Estudiantes que respondieron: 2558
Chi cuadrado: 220.252
Valor p < 0.001
Se rechaza H0: si hay relacion entre las variables
V de Cramer: 0.293


In [5]:
si = con_internet[con_internet['internet'] == 'Si']['global']
no = con_internet[con_internet['internet'] == 'No']['global']
print('Con internet:', len(si), '| sin internet:', len(no))
print('Promedio con internet:', round(si.mean(), 1))
print('Promedio sin internet:', round(no.mean(), 1))
p3 = prueba_t(si, no)
d3 = d_cohen(si, no)

Con internet: 1721 | sin internet: 837
Promedio con internet: 261.8
Promedio sin internet: 225.8
Estadistico t: 17.976
Valor p < 0.001
Se rechaza H0: si hay diferencia entre los grupos
d de Cohen: 0.703 (efecto mediano)


In [6]:
estratos = ['Estrato 1', 'Estrato 2', 'Estrato 3', 'Estrato 4', 'Estrato 5', 'Estrato 6']
con_estrato = tabla[tabla['estrato'].isin(estratos)]
display(con_estrato.groupby('estrato')['global'].agg(['count', 'mean', 'median']).round(1))
p4 = prueba_kruskal([con_estrato[con_estrato['estrato'] == e]['global'] for e in estratos])

,count,mean,median
estrato,,,
Estrato 1,863,238.8,235.0
Estrato 2,820,249.9,246.0
Estrato 3,470,265.3,268.0
Estrato 4,161,282.5,291.0
Estrato 5,49,289.7,304.0
Estrato 6,34,279.1,296.0


H de Kruskal-Wallis: 140.414
Valor p < 0.001
Epsilon cuadrado: 0.059
Se rechaza H0: al menos un grupo es distinto


## Control: la brecha por zona dentro de cada estrato

Si la brecha rural-urbana fuera solo un efecto del estrato, al comparar
rurales y urbanos del mismo estrato la diferencia deberia desaparecer.
Lo revisamos en los estratos 1 a 3, que son los que tienen suficientes
estudiantes rurales.

In [7]:
por_estrato = (con_estrato[con_estrato['estrato'].isin(estratos[:3])]
               .groupby(['estrato', 'zona'])['global'].agg(['count', 'mean'])
               .unstack('zona'))
por_estrato['brecha'] = por_estrato[('mean', 'URBANO')] - por_estrato[('mean', 'RURAL')]
display(por_estrato.round(1))

count          mean        brecha
zona      RURAL URBANO  RURAL URBANO       
estrato                                    
Estrato 1   265    598  227.9  243.7   15.8
Estrato 2   161    659  224.0  256.3   32.2
Estrato 3    64    406  223.8  271.9   48.1

## Control: la brecha por zona segun internet

Lo mismo con el internet: comparamos rurales y urbanos que tienen internet,
y rurales y urbanos que no tienen.

In [8]:
por_internet = (con_internet.groupby(['internet', 'zona'])['global']
                .agg(['count', 'mean']).unstack('zona'))
por_internet['brecha'] = por_internet[('mean', 'URBANO')] - por_internet[('mean', 'RURAL')]
display(por_internet.round(1))

count          mean        brecha
zona     RURAL URBANO  RURAL URBANO       
internet                                  
No         351    486  219.1  230.6   11.5
Si         261   1460  237.6  266.2   28.6

## Conclusion

El analisis esta en docs/02_hipotesis.md.